In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lpad, concat_ws, sum as spark_sum, max as spark_max, expr

spark = SparkSession.builder.getOrCreate()

# 1) Leer fichero
df = spark.read.option("header", True) \
               .option("sep", ";") \
               .option("inferSchema", True) \
               .csv("work/calidad_aire_datos_meteo_mes.csv")

# 2) Filtrar magnitud 89
df_prec = df.filter(col("MAGNITUD") == 89)

# 3) Crear fecha YYYY-MM-DD
df_prec = df_prec.withColumn(
    "FECHA",
    concat_ws(
        "-",
        col("ANO").cast("string"),
        lpad(col("MES").cast("string"), 2, "0"),
        lpad(col("DIA").cast("string"), 2, "0")
    )
)

# 4) Pasar H01..H24 y V01..V24 a filas
df_horas = df_prec.select(
    "FECHA",
    "MUNICIPIO",
    "ESTACION",
    expr("""
        stack(24,
            '01', H01, V01,
            '02', H02, V02,
            '03', H03, V03,
            '04', H04, V04,
            '05', H05, V05,
            '06', H06, V06,
            '07', H07, V07,
            '08', H08, V08,
            '09', H09, V09,
            '10', H10, V10,
            '11', H11, V11,
            '12', H12, V12,
            '13', H13, V13,
            '14', H14, V14,
            '15', H15, V15,
            '16', H16, V16,
            '17', H17, V17,
            '18', H18, V18,
            '19', H19, V19,
            '20', H20, V20,
            '21', H21, V21,
            '22', H22, V22,
            '23', H23, V23,
            '24', H24, V24
        ) as (HORA, PRECIP, VALIDACION)
    """)
)

# 5) Solo horas con validación V
df_validas = df_horas.filter(
    (col("VALIDACION") == "V") & col("PRECIP").isNotNull()
)

# 6) Suma diaria por estación
df_diario = df_validas.groupBy("FECHA", "MUNICIPIO", "ESTACION") \
    .agg(spark_sum("PRECIP").alias("PRECIPITACION_TOTAL"))

# 7) Máximo de precipitación por día
df_max_dia = df_diario.groupBy("FECHA") \
    .agg(spark_max("PRECIPITACION_TOTAL").alias("MAX_PRECIP_DIA"))

# 8) Join con alias para evitar ambigüedad
d = df_diario.alias("d")
m = df_max_dia.alias("m")

resultado = d.join(
    m,
    (col("d.FECHA") == col("m.FECHA")) &
    (col("d.PRECIPITACION_TOTAL") == col("m.MAX_PRECIP_DIA")),
    "inner"
).select(
    col("d.FECHA").alias("FECHA"),
    col("d.MUNICIPIO").alias("MUNICIPIO"),
    col("d.ESTACION").alias("ESTACION"),
    col("d.PRECIPITACION_TOTAL").alias("PRECIPITACION_TOTAL")
).orderBy("FECHA")

print("Mayor precipitación por cada día:")
resultado.show(truncate=False)

# 9) Máximo global del periodo
max_total = resultado.agg(
    spark_max("PRECIPITACION_TOTAL").alias("MAX_TOTAL")
).alias("mt")

r = resultado.alias("r")

resultado_max_total = r.join(
    max_total,
    col("r.PRECIPITACION_TOTAL") == col("mt.MAX_TOTAL"),
    "inner"
).select(
    col("r.FECHA"),
    col("r.MUNICIPIO"),
    col("r.ESTACION"),
    col("r.PRECIPITACION_TOTAL")
)

print("Mayor precipitación diaria de todo el periodo:")
resultado_max_total.show(truncate=False)

AnalysisException: Column FECHA#74 are ambiguous. It's probably because you joined several Datasets together, and some of these Datasets are the same. This column points to one of the Datasets but Spark is unable to figure out which one. Please alias the Datasets with different names via `Dataset.as` before joining them, and specify the column using qualified name, e.g. `df.as("a").join(df.as("b"), $"a.id" > $"b.id")`. You can also set spark.sql.analyzer.failAmbiguousSelfJoin to false to disable this check.